In [4]:
#TASK 1 - LOAD THE DATA
import numpy as np
from PIL import Image
import os

DATASET_PATH = "dataset"

In [5]:
# Names of 10 character classes

class_names = [
    "bart_simpson",
    "charles_montgomery_burns",
    "homer_simpson",
    "krusty_the_clown",
    "lisa_simpson",
    "marge_simpson",
    "milhouse_van_houten",
    "moe_szyslak",
    "ned_flanders",
    "principal_skinner"
]


In [6]:
def load_images(data_dir, image_mode):
    """Load JPEG images from class subfolders into flattened, normalised
    NumPy arrays, with integer labels based on folder order."""
    images = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)

        for filename in sorted(os.listdir(class_dir)):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                image_path = os.path.join(class_dir, filename)
                with Image.open(image_path) as image:
                    image = image.convert(image_mode)
                    image_array = np.asarray(image, dtype=np.float32).flatten() / 255.0
                images.append(image_array)
                labels.append(label)
    X = np.array(images)
    y = np.array(labels)
    return X, y

In [7]:
#Load grayscale training and test data
X_train_gray, y_train_gray = load_images(
    os.path.join(DATASET_PATH, "grayscale", "train"), "L"
)
X_test_gray, y_test_gray = load_images(
    os.path.join(DATASET_PATH, "grayscale", "test"), "L"
)

#Load RGB training and test data
X_train_rgb, y_train_rgb = load_images(
    os.path.join(DATASET_PATH, "rgb", "train"), "RGB"
)
X_test_rgb, y_test_rgb = load_images(
    os.path.join(DATASET_PATH, "rgb", "test"), "RGB"
)

In [8]:
# Splitin off a validation set from the training data
np.random.seed(1234)
valid_size = X_test_gray.shape[0]
indices = np.random.choice(X_train_gray.shape[0], valid_size, replace=False)

X_valid_gray = X_train_gray[indices]
y_valid_gray = y_train_gray[indices]
X_train_gray = np.delete(X_train_gray, indices, axis=0)
y_train_gray = np.delete(y_train_gray, indices, axis=0)

X_valid_rgb = X_train_rgb[indices]
y_valid_rgb = y_train_rgb[indices]
X_train_rgb = np.delete(X_train_rgb, indices, axis=0)
y_train_rgb = np.delete(y_train_rgb, indices, axis=0)


In [9]:
# Checking size of each dataset
print("Grayscale:", X_train_gray.shape, X_valid_gray.shape, X_test_gray.shape)
print("RGB:      ", X_train_rgb.shape, X_valid_rgb.shape, X_test_rgb.shape)

Grayscale: (6000, 784) (2000, 784) (2000, 784)
RGB:       (6000, 2352) (2000, 2352) (2000, 2352)


In [10]:
# TASK 2

class BinaryPerceptron:
 
    def __init__(self, n_inputs, alpha=0.01):
        self.weights = np.full(n_inputs, 0.1)
        self.bias = 0.1
        self.alpha = alpha
 
    def net_input(self, x):
        return np.dot(x, self.weights) + self.bias
 
    def predict(self, x):
        return 1 if self.net_input(x) >= 0 else 0
 
    def apply_learning_rule(self, x, y):
        # Applying the perceptron update rule: comparing my prediction to the true label and only adjusting weights/bias when they differ
        g = self.predict(x)
        self.weights = self.weights + self.alpha * (y - g) * x
        self.bias = self.bias + self.alpha * (y - g)

In [11]:

def train(model, X_train, y_train, X_valid, y_valid, epochs=10):
 
    for epoch in range(epochs):
 
        # Shuffling the training data each epoch so I'm not feeding it examples in the same class-grouped order every time
        indices = np.random.permutation(len(X_train))
 
        for i in indices:
            model.apply_learning_rule(X_train[i], y_train[i])
 
        correct = 0
 
        for i in range(len(X_valid)):
            if model.predict(X_valid[i]) == y_valid[i]:
                correct += 1
 
        accuracy = correct / len(X_valid)
 
        print(f"Epoch {epoch + 1} - Validation accuracy: {accuracy:.2f}")

In [12]:
class MultiClassPerceptron:
 
    def __init__(self, n_inputs, n_classes=10, alpha=0.01):
        self.perceptrons = []
 
        for i in range(n_classes):
            self.perceptrons.append(
                BinaryPerceptron(n_inputs, alpha)
            )
 
    def predict(self, x):
        # Getting a raw score from each of the 10 perceptrons, then picking whichever class's perceptron is most confident
        scores = []
 
        for perceptron in self.perceptrons:
            scores.append(perceptron.net_input(x))
 
        return np.argmax(scores)
 
    def apply_learning_rule(self, x, y):
 
        for class_index in range(10):
 
            # Turning the multi-class label into a binary target: 1 for the perceptron matching the true class, 0 for the rest
            if class_index == y:
                target = 1
            else:
                target = 0
 
            self.perceptrons[class_index].apply_learning_rule(
                x, target
            )
 